# Phase 2 : Comparaison de modeles de Machine Learning
Dans ce notebook, on compare trois algorithmes après avoir nettoyé le texte. Les resultats sont affiches sous forme de tableaux pour plus de clarte.

In [ ]:
import pandas as pd
import os
import sys
import pickle 
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

sys.path.append(os.path.abspath('..'))
from src.data_loader import load_yelp_sample
from src.preprocessing import clean_text  # <-- IMPORT DU PREPROCESSING

# Chargement des donnees
df = load_yelp_sample('../data/raw/review.json', n_rows=50000)
df['label'] = (df['stars'] <= 2).astype(int)

# Nettoyage du texte
print("Nettoyage du texte en cours...")
df['text'] = df['text'].apply(clean_text)

# Vectorisation (Unigrammes + Bigrammes)
print("Vectorisation...")
vec = TfidfVectorizer(max_features=5000, ngram_range=(1,2), stop_words='english')
X = vec.fit_transform(df['text'])
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

Chargement de 50000 lignes depuis ../data/raw/review.json...
Nettoyage du texte en cours...
Vectorisation...


### Modele 1 : Regression Logistique

In [7]:
model_lr = LogisticRegression(class_weight='balanced', max_iter=1000)
model_lr.fit(X_train, y_train)

report_lr = classification_report(y_test, model_lr.predict(X_test), output_dict=True)
df_lr = pd.DataFrame(report_lr).transpose()
display(df_lr.round(2))

,precision,recall,f1-score,support
0,0.97,0.91,0.94,7710.00
1,0.75,0.91,0.82,2290.00
accuracy,0.91,0.91,0.91,0.91
macro avg,0.86,0.91,0.88,10000.00
weighted avg,0.92,0.91,0.91,10000.00


### Modele 2 : Random Forest

In [8]:
model_rf = RandomForestClassifier(n_estimators=100, max_depth=20, class_weight='balanced')
model_rf.fit(X_train, y_train)

report_rf = classification_report(y_test, model_rf.predict(X_test), output_dict=True)
df_rf = pd.DataFrame(report_rf).transpose()
display(df_rf.round(2))

,precision,recall,f1-score,support
0,0.94,0.91,0.92,7710.00
1,0.72,0.81,0.77,2290.00
accuracy,0.89,0.89,0.89,0.89
macro avg,0.83,0.86,0.85,10000.00
weighted avg,0.89,0.89,0.89,10000.00


### Modele 3 : Naive Bayes

In [9]:
model_nb = MultinomialNB()
model_nb.fit(X_train, y_train)

report_nb = classification_report(y_test, model_nb.predict(X_test), output_dict=True)
df_nb = pd.DataFrame(report_nb).transpose()
display(df_nb.round(2))

# Sauvegarde des modèles pour le Notebook 3
os.makedirs('../models', exist_ok=True)
pickle.dump(model_lr, open('../models/classic_model.pkl', 'wb'))
pickle.dump(vec, open('../models/vectorizer.pkl', 'wb'))
print("Modèles sauvegardés dans le dossier 'models'.")

,precision,recall,f1-score,support
0,0.90,0.96,0.93,7710.00
1,0.82,0.66,0.73,2290.00
accuracy,0.89,0.89,0.89,0.89
macro avg,0.86,0.81,0.83,10000.00
weighted avg,0.89,0.89,0.89,10000.00


Modèles sauvegardés dans le dossier 'models'.
